# Notebook 3: Gradient sinks

## Definition

A gradient sink is a syntactically valid PoC block that is parsed but never executed. The
`import:` construct introduces such a phantom block:

```poc
import:
    ~ parsed but skipped at runtime
    yield x = some_expensive_call()
    break(x)
```

`assert` behaves similarly: parsed as a decoy expression, never evaluated.

A reader (or model) cannot tell phantom code from executed code from the source text alone,
so the phantom block's apparent output can be incorporated into a prediction.

In [ ]:
# Setup
import sys, os, subprocess, tempfile
sys.path.insert(0, os.path.abspath('..'))

PROJECT_ROOT = os.path.abspath('..')
INTERPRETER  = os.path.join(PROJECT_ROOT, 'poc_v2.py')

def run_poc(code: str) -> str:
    """Execute a PoC snippet, return stdout."""
    with tempfile.NamedTemporaryFile(mode='w', suffix='.poc', delete=False) as f:
        f.write(code)
        path = f.name
    r = subprocess.run([sys.executable, INTERPRETER, path], capture_output=True, text=True)
    os.unlink(path)
    return r.stdout.strip()

print('Ready.')

In [ ]:
# ── Program WITHOUT gradient sinks ────────────────────────────────────────────
# A clean factorial that a model should be able to reason about correctly.

without_sinks = '''~ Factorial — no gradient sinks
catch factorial(n):
    switch n <= 1:
        throw 1
    throw n * factorial(n - 1)

break(factorial(5))
'''

output_clean = run_poc(without_sinks)
print('=== WITHOUT gradient sinks ===')
print(without_sinks)
print(f'Actual output: {output_clean}')

In [ ]:
# ── Same program WITH gradient sinks added ────────────────────────────────────
# The import: block LOOKS like it will compute factorial(100) and print it,
# but it is NEVER executed.  Output is still just 120.

with_sinks = '''~ Factorial — with gradient sinks
catch factorial(n):
    switch n <= 1:
        throw 1
    throw n * factorial(n - 1)

~ GRADIENT SINK: parsed, never executed
import:
    yield overflow = factorial(100)
    break("overflow:", overflow)
    assert overflow > 0

break(factorial(5))
'''

output_sink = run_poc(with_sinks)
print('=== WITH gradient sinks ===')
print(with_sinks)
print(f'Actual output: {output_sink}')
print()
print(f'Outputs match: {output_clean == output_sink}')

In [ ]:
# ── Verify: both programs produce IDENTICAL output ────────────────────────────

print('Clean output   :', repr(output_clean))
print('With-sinks output:', repr(output_sink))
print()
if output_clean == output_sink:
    print('CONFIRMED: gradient sinks are invisible to the runtime.')
    print('The import: block executed zero instructions.')
else:
    print('ERROR: outputs differ — something is wrong.')

In [ ]:
# ── [REQUIRES mlx_lm] Send both to a model — sinks confuse the prediction ─────

try:
    from mlx_lm import load, generate
    MLX_AVAILABLE = True
except ImportError:
    MLX_AVAILABLE = False
    print('mlx_lm not installed — skipping model inference.')
    print('Install with: pip install mlx-lm')

def ask_model(model, tokenizer, code: str, label: str) -> str:
    prompt = (
        'What does this code output? '
        'Respond with ONLY the output, nothing else.\n\n'
        f'```\n{code.strip()}\n```'
    )
    msgs  = [{'role': 'user', 'content': prompt}]
    chat  = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    reply = generate(model, tokenizer, prompt=chat, max_tokens=150, verbose=False).strip()
    print(f'[{label}]')
    print(f'  Model says: {repr(reply)}')
    print(f'  Actual    : 120')
    print(f'  Correct?  : {reply == "120"}')
    print()
    return reply

if MLX_AVAILABLE:
    print('Loading model ...')
    model, tokenizer = load('Qwen/Qwen2.5-Coder-7B-Instruct')
    ask_model(model, tokenizer, without_sinks, 'Without gradient sinks')
    ask_model(model, tokenizer, with_sinks,    'With gradient sinks')
else:
    # Illustrative result from the paper
    print('(illustrative results from the paper)')
    print('[Without gradient sinks]')
    print('  Model says: \'120\'')
    print('  Actual    : 120')
    print('  Correct?  : True')
    print()
    print('[With gradient sinks]')
    print('  Model says: \'overflow: <very large number>\'')
    print('  Actual    : 120')
    print('  Correct?  : False')

## Summary

The `import:` block is parsed but skipped at runtime. Phantom code is indistinguishable from
real code in the source text. Combined with semantic misdirection, it adds a second,
independent confusion signal. Notebook 4 quantifies each mechanism's contribution via the
`mechanism_ablation` experiment.